In [14]:
!pip install --upgrade google-cloud-aiplatform google-adk litellm requests

  Using cached google_cloud_aiplatform-1.158.0-py2.py3-none-any.whl.metadata (50 kB)
  Using cached google_adk-2.3.0-py3-none-any.whl.metadata (16 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 5.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of google-genai to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.3 MB/s eta 0:00:00
INFO: pip is still looking at multiple versions of google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of google-cloud-storage to determine which version is compatible with other requirements. This could 

In [1]:
!pip install google-adk[extensions]

INFO: pip is looking at multiple versions of litellm to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━

In [ ]:
import os
import requests
from typing import Tuple, Dict, Any, Optional, List

# Importations du Gemini Agent Development Kit (ADK) et de LiteLLM
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY", "YOUR_GEMINI_KEY")
os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY", "TA_CLE_CLAUDE")
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "YOUR_MAPS_API_KEY")

In [3]:
def get_lat_lon(address: str) -> Optional[Tuple[float, float]]:
    """
    Convert a textual address or city name into latitude and longitude
    using the Google Maps Geocoding API.

    Args:
        address (str): The string representing the location (e.g., "Los Angeles, CA").

    Returns:
        Optional[Tuple[float, float]]: A tuple containing (latitude, longitude)
        if successful. Returns None if an error occurs.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address,
        "key": GOOGLE_MAPS_API_KEY
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        if data["status"] == "OK":
            location = data["results"][0]["geometry"]["location"]
            return location["lat"], location["lng"]
        else:
            print(f"Geocoding error: {data['status']}")
            return None

    except requests.RequestException as e:
        print(f"API Request failed: {e}")
        return None

In [4]:
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended weather forecast from the U.S. National Weather Service API
    based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast dictionaries.
        Returns None if data is unavailable or an error occurs.
    """
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    headers = {"User-Agent": "(myweatheragent.com, contact@example.com)"}

    try:
        response = requests.get(points_url, headers=headers)
        response.raise_for_status()
        points_data = response.json()

        forecast_url = points_data["properties"]["forecast"]

        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()
        forecast_data = forecast_response.json()

        # Transformation légère pour s'assurer du format List[Dict[str, str]] attendu
        periods = forecast_data["properties"]["periods"]
        cleaned_periods = []
        for period in periods:
            cleaned_periods.append({
                "name": str(period.get("name", "")),
                "temperature": f"{period.get('temperature', '')} {period.get('temperatureUnit', '')}",
                "detailedForecast": str(period.get("detailedForecast", ""))
            })

        return cleaned_periods

    except requests.RequestException as e:
        print(f"NWS API Request failed: {e}")
        return None

In [5]:
WEATHER_AGENT_INSTRUCTIONS = """You are Pat, a friendly weather agent. Your job is to provide accurate weather forecasts for US cities.
To answer a user's request, follow these steps strictly:
1. Always use the `get_lat_lon` tool first to find the exact latitude and longitude of the city requested by the user.
2. Pass those exact coordinates into the `get_extended_weather_forecast` tool to get the current weather data.
3. Summarize the weather forecast clearly and cheerfully for the user, mentioning the temperature and general conditions.
Only use the tools provided to look up information."""

# Liste des outils passés aux agents
weather_tools = [get_extended_weather_forecast, get_lat_lon]

In [6]:
# Configuration de Pat avec le modèle Gemini natif de l'ADK
weather_agent = Agent(
    name="Pat",
    model="gemini-2.5-flash",
    description="Pat the Friendly Weather Agent.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=weather_tools
)

In [ ]:
# Configuration de Pat avec un modèle tiers via l'intégration LiteLlm de l'ADK
claude_weather_agent = Agent(
    name="Pat-Claude",
    model=LiteLlm(model="anthropic/claude-3-5-sonnet-20241022"),
    description="Pat the Friendly Weather Agent (powered by Claude).",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=weather_tools
)

In [11]:
import os
from vertexai.preview import reasoning_engines

# Initialisation des applications ADK
if 'app' not in locals() and 'app' not in globals():
    app = reasoning_engines.AdkApp(agent=weather_agent)

# if 'app_claude' not in locals() and 'app_claude' not in globals():
#     app_claude = reasoning_engines.AdkApp(agent=claude_weather_agent)

# Configuration des sessions de test
test_user = "test-runner"

try:
    session_gemini_obj = app.create_session(user_id=test_user)
    # Correction : On extrait l'ID de manière sûre qu'il s'agisse d'un dictionnaire ou d'un objet
    session_gemini_id = session_gemini_obj.get("session_id") if isinstance(session_gemini_obj, dict) else getattr(session_gemini_obj, "id", None)
except Exception as e:
    print(f"Impossible de générer une session Gemini Cloud : {e}")
    session_gemini_id = "fallback-session-id"

# try:
#     session_claude_obj = app_claude.create_session(user_id=test_user)
#     session_claude_id = session_claude_obj.get("session_id") if isinstance(session_claude_obj, dict) else getattr(session_claude_obj, "id", None)
# except Exception as e:
#     print(f"Impossible de générer une session Claude Cloud : {e}")
#     session_claude_id = "fallback-session-id"

test_cities = [
    "New York, NY",
    "Seattle, WA",
    "Miami, FL"
]

print("\n=== TEST DE L'AGENT NATIF GEMINI (Pat via stream_query) ===")
for city in test_cities:
    prompt = f"Hi Pat! What is the weather like in {city}?"
    print(f"\n[User]: {prompt}")
    try:
        response_text = ""
        # Consommation du stream
        for event in app.stream_query(
            user_id=test_user,
            session_id=session_gemini_id,
            message=prompt
        ):
            if "content" in event and "parts" in event["content"]:
                for part in event["content"]["parts"]:
                    if "text" in part:
                        response_text += part["text"]

        print(f"[{weather_agent.name}]:\n{response_text}")
    except Exception as e:
        print(f"Erreur avec Gemini pour {city}: {e}")

print("\n" + "="*60 + "\n")

# print("=== TEST DE L'AGENT TIERS VIA LITELLM (Pat-Claude via stream_query) ===")
# for city in test_cities:
#     prompt = f"Hi Pat! What is the weather like in {city}?"
#     print(f"\n[User]: {prompt}")
#     try:
#         response_text = ""
#         for event in app_claude.stream_query(
#             user_id=test_user,
#             session_id=session_claude_id,
#             message=prompt
#         ):
#             if "content" in event and "parts" in event["content"]:
#                 for part in event["content"]["parts"]:
#                     if "text" in part:
#                         response_text += part["text"]

#         print(f"[{claude_weather_agent.name}]:\n{response_text}")
#     except Exception as e:
#         print(f"Erreur avec Claude pour {city}: {e}")


=== TEST DE L'AGENT NATIF GEMINI (Pat via stream_query) ===

[User]: Hi Pat! What is the weather like in New York, NY?


/usr/local/lib/python3.12/dist-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


[Pat]:
Hello there! The weather in New York, NY today is mostly cloudy with a high near 76°F, but temperatures will be falling to around 71°F in the afternoon. There's an 80% chance of rain showers before 2 PM, followed by showers and thunderstorms, some of which could bring heavy rain.

[User]: Hi Pat! What is the weather like in Seattle, WA?
[Pat]:
Good news! The weather in Seattle, WA today is sunny with a high of 82 degrees Fahrenheit. There will be a light breeze from the North between 2 to 10 mph. Enjoy the beautiful day!

[User]: Hi Pat! What is the weather like in Miami, FL?
[Pat]:
Hello there! In Miami, FL today, it's going to be a sunny day with a high near 90°F. The heat index could reach as high as 105°F, and there might be some patchy smoke. A gentle southeast wind will be blowing at 5 to 9 mph. Enjoy the beautiful weather!


